This is the classification algorithm as described in the worksheet `classification.ipynb`.  It relies on two prior worksheets.  To make it up to date with all current placements, run `mike_assemble_data.ipynb` to fetch the latest data, then `mike_create_classification.ipynb`.  The latter no longer does classification.  Instead it sets the various parameters (number of types etc) and filters the data (so each applicant has a single placment).

The initialization code reads various files that were created this filtering code and uses them here.

In [1]:
## method based on the article in classification.ipynb
#  uses preprocessed information from mike_create_classification,ipy
using HTTP, JSON, PrettyTables, JLD, DotEnv,  Random, Dates
include("functions/type_allocation_flexible.jl")
include("functions/type_allocation_base.jl")
cfg = DotEnv.config("../.env")
files_path = cfg["files_path"]
institutions = load(files_path*"institutions.jld")["institutions"]
academic_list = load(files_path*"academic_list.jld")["academic_list"]
adjacency = load(files_path*"out.jld")["out"];
institution_mapping = load(files_path*"institution_mapping.jld")["institution_mapping"]
reverse_mapping = load(files_path*"reverse_mapping.jld")["reverse_mapping"]
classification_properties = load(files_path*"/classification_properties.jld")["classification_properties"];

In [2]:
cp = classification_properties

Dict{String, Any} with 7 entries:
  "data_loaded"      => DateTime("2025-07-22T03:35:38.437")
  "number_of_types"  => 5
  "labels"           => Any["Public Sector", "Private Sector", "Postdocs", "Lec…
  "number_of_sinks"  => 6
  "algorithm_run_id" => 7
  "sink_alloc"       => Any[6, 6, 6, 6, 6, 6, 6, 6, 6, 6  …  11, 11, 11, 11, 11…
  "counter"          => Any[226, 335, 801, 614, 674, 814]

In [3]:
# check sanity - compare with the total in classification_filter
size(adjacency)

(4729, 1265)

In [4]:
# sanity check - verify the total number of placements is euqal to the total of all the cells in the adjacecny matrxi
sum(adjacency)

21425

In [5]:
C = zeros(Int32,cp["number_of_types"]+cp["number_of_sinks"],
    length(institutions))
T = zeros(Int32, length(academic_list), cp["number_of_types"]);

In [6]:
# sinks have been hard coded - start by assigning all academic institutions to community 1
for i in 1:length(academic_list)
    T[i,1] = 1
    C[1,i] = 1
end
#now all academic institutions are in the same tier

In [10]:
#forgot to allocate the sinks
for i in length(academic_list) + 1:length(institutions)
    C[cp["sink_alloc"][i-length(academic_list)], i] = 1
end

In [11]:
sum(C, dims = 2)

11×1 Matrix{Int64}:
 1265
    0
    0
    0
    0
  226
  335
  801
  614
  674
  814

In [8]:
count(x -> x == 6,cp["sink_alloc"])

226

In [12]:
C*adjacency*T

11×5 Matrix{Int32}:
 13908  0  0  0  0
     0  0  0  0  0
     0  0  0  0  0
     0  0  0  0  0
     0  0  0  0  0
  1953  0  0  0  0
  1977  0  0  0  0
   767  0  0  0  0
   524  0  0  0  0
   933  0  0  0  0
  1363  0  0  0  0

In [11]:
function likelihood(adjacency, c, t)
    # the number of institutions in each tier
    # the number of academic institions in each tier
    l = 0.0
    for i in 1:11,j in 1:5
        l +=  -adjacency[i,j]*(log(max(adjacency[i,j]/(max(c[i]*t[j],1)),.0001)) - 1)
    end
    return l
end
    

likelihood (generic function with 1 method)

In [12]:
function best_alloc(C,T,adjacency, iterations = 10) 
    c = sum(C, dims = 2)
    t = sum(T, dims=1)
    start = likelihood(C*adjacency*T, c, t)
    new_l = start
    sq = [C, T, start]
    println("Starting value: ", start)
    println(C*adjacency*T)
    n = 0
    while n < iterations
        #designed originally for many suffles at once - since we are only doing one at a time,
        shuffle = []
        to_tier = []
        new_C = zeros(Int32,size(C,1),size(C,2))
        new_T = zeros(Int32, size(T,1),size(T,2))
        for j in 1:1
            push!(shuffle, rand(1:size(T,1)))
            push!(to_tier, rand(1:size(T,2)))
        end
    
        for i in 1:size(C,2), j in 1:size(C,1)
            index = findfirst(x -> x == i, shuffle)
            if !isnothing(index)
                if j == to_tier[index] 
                    new_C[j, i] = 1
                    new_T[i, j] = 1
                else
                    new_C[j, i] = 0
                    if j < size(T,2) + 1
                        new_T[i, j] = 0
                    end
                end
            else
                new_C[j,i] = sq[1][j,i]
                if i < size(T,1) + 1
                    if j < size(T,2) + 1
                        new_T[i, j] = sq[2][i,j]
                    end
                end
            end
        end
      
        #println(new_C*adjacency*new_T)
        c = sum(new_C, dims = 2)
        t = sum(new_T, dims = 1)
        new_l = likelihood(new_C*adjacency*new_T,c,t)
        if isinteger(n/10000)
            println(sq[3])
        end
        
        if new_l < sq[3]
            sq[3] = new_l
            sq[1] = new_C
            sq[2] = new_T
        end
        n += 1
    end
    println("Iteration ", n, ": " , sq[3])           
    return sq
end
        

best_alloc (generic function with 2 methods)

In [13]:
@time sq = best_alloc(C,T,adjacency, 300000);



Starting value: 132506.08460847655
Int32[13908 0 0 0 0; 0 0 0 0 0; 0 0 0 0 0; 0 0 0 0 0; 0 0 0 0 0; 1953 0 0 0 0; 1977 0 0 0 0; 767 0 0 0 0; 524 0 0 0 0; 933 0 0 0 0; 1363 0 0 0 0]
132506.08460847655
93990.33085154339
90973.12027560391
90365.76790425507
90200.47729795401
90134.3325390644
90122.17172251208
90113.9352081927
90104.66086205406
90102.07790775207
90095.0621327742
90089.07544902536
90086.2167241375
90080.44177989411
90077.1888501395
90077.07429359875
90076.86444059816
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
90076.83860610652
Iteration 300000: 90076.83860610652
1836.411775 seconds (30.66 G allocations: 548.828 GiB, 2.60% gc time, 0.05% compilation time)


In [14]:
p = sq[1]*adjacency*sq[2]

11×5 Matrix{Int32}:
 1402   73  1137  221  1486
  107  460    36   92    77
  300   16  1638   62   525
  623  104   375  682   692
  566   38  1444  165  1587
  396   55   716  108   678
  349   54   926   99   549
  219   66   161   84   237
  163  107    65   82   107
  192   58   319   62   302
  422  168   250  168   355

In [15]:
# verify with sum check
sum(p)

21425

In [16]:
function sort_by_diag(p)
   """
        Helper function for sorting an unsorted placement matrix
        finds the biggest diagonal element of a (non-diagonal) matrix
        returns the index of the biggest element, and a copy of the matrix with the
        column that corresponds with that index reset to 0
    """
    num_types = size(p)[2]
    p_normalized = p./sum(p, dims=2)
    # println(repr("text/plain", p_normalized))
    max_diag = 1
    for k in 1:num_types
        if isnan(p_normalized[k,k])
            continue
        else 
            if isnan(p_normalized[max_diag, max_diag])
                max_diag = k
            else 
                if isless(p_normalized[max_diag, max_diag],p_normalized[k,k])  
                    max_diag = k
                end
            end
        end
    end
    p_normalized[:,max_diag] .= 0
    return max_diag, p_normalized
end

sort_by_diag (generic function with 1 method)

In [17]:
function sort_matrix(est_mat)
    o = zeros(Int32, size(est_mat)[1])
    p = zeros(Int32, size(est_mat)[1], size(est_mat)[2])
    placement_rates = zeros(Int32, size(est_mat))
    for i in 1:size(est_mat)[1]
        for j in 1:size(est_mat)[2]
            p[i,j] = est_mat[i,j]
        end
    end
    #println("\ndebug type_allocation base line 335 ", p)
    for k in 1:size(p)[2]
        max_diag, p = sort_by_diag(p);
        o[k] = max_diag
    end
    for k in size(est_mat)[2]+1:size(est_mat)[1]
        o[k] = k
    end
    for i in 1:size(est_mat)[1]
        for j in 1:size(est_mat)[2]
            #println(i, " ", j)
            placement_rates[i, j] = est_mat[o[i], o[j]]
        end
    end
    return placement_rates, o
end

sort_matrix (generic function with 1 method)

In [71]:
# re-order the adjacency matrix to turn it into a tier format
# based on method in the theory paper
# p2 is the ordered adjacency matrix, o is the order translateion as described above
p2, o = sort_matrix(p);

In [26]:
p2

11×5 Matrix{Int32}:
 1638   525   300   62   16
 1444  1587   566  165   38
 1137  1486  1402  221   73
  375   692   623  682  104
   36    77   107   92  460
  716   678   396  108   55
  926   549   349   99   54
  161   237   219   84   66
   65   107   163   82  107
  319   302   192   62   58
  250   355   422  168  168

In [72]:
# save the estimated matrix
save(files_path*"/placement_rates.jld", "placement_rates", p2)

In [20]:
unordered = sum(sq[1], dims = 2)

11×1 Matrix{Int64}:
 198
 621
  24
 345
  77
 226
 335
 801
 614
 674
 814

In [63]:
ordered_counts = []
for i in 1:cp["number_of_types"]
    push!(ordered_counts, unordered[o[i]])
end
for i in cp["number_of_types"]+1:cp["number_of_types"]+cp["number_of_sinks"]
    push!(ordered_counts, unordered[i])
end
        ordered_counts

11-element Vector{Any}:
  24
  77
 198
 345
 621
 226
 335
 801
 614
 674
 814

In [15]:
save(files_path*"/institution_counts.jld", "institution_counts", ordered_counts)

LoadError: UndefVarError: `ordered_counts` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [23]:
# check for data consistency
sum(p2)

21425

In [1]:
# For listing of academic institutions in different tiers
# change the number in the o() function to the number of the tier you want to list
for i in 1:size(adjacency,1)
    if sq[1][o[6],i] == 1
        println(institutions[i])
    end
end

LoadError: UndefVarError: `adjacency` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Summary of results
The next segment does two things.  It creates a nice tabular represetntation of the adjacency matrxi that can be used in tex files. The second part is to save the estimated type allocation in a fashion that can be used in subsequent estimation.

The basic classifier result is given by `sq`, an array with 3 elements.  The first element `sq[1]` is the basic classification array with dimension `number_of_types + number_of_sinks` by the total number of institutions (`length(institutions)`).

The ordered adjacency matrix is given by `p2`, with dimension `number_of_types + number_of_sinks` by `number_of_types`.  Two facts to keep in mind, the classification matrix `sq[1]` is unordered.  The translation to the tier based order is given by the array `o`.  This array translates an unordered type into a tier based ordered type.  For example, tier 1 could be given by anything that is allocated to type 3 by `sq[1]`.  So `o[3]` would have value 1 in that case.

The the right tier based allocation is done the following way:  
1. choose a column in `sq[1]`, say column `j`
2. the name of the institution associated with that column is given by `institutions[j]`;
3. find each of the rows for which cell $(i,j)$ has value 1.  There might be more than one, but suppose for simplicity  there is just one, say row $i$.
4. The tier of institution $j$  in that case would be `o[i]`.

The reason that there might be more than one assignment for each institution is that they will generally play more than one role.  For example, an institution from tier 1 which hires graduates as assistant professors might hire post-docs so that it would also be classified in tier `post-doc`.

In the classification library `id_to_type_api.json` the recorded classification is the lowest tier in which an institution is classified.

We begin with a construction of the classification library.

In [58]:
type_dictionary = []
for i in 1:length(academic_list)
    institution_id = reverse_mapping[institutions[i]]
    t = findall(sq[1][:,i].==1)
    push!(type_dictionary, Dict("name" => institutions[i],
                "institution_id" => institution_id,
                "type" => o[t[1]]))
end     

In [61]:
open(files_path*"id_to_type_api.json", "w") do f
    write(f, JSON.json(type_dictionary))
end;

In [60]:
## show tiers using dictionary
for t in type_dictionary
    if t["type"] == 1
        println(t["name"])
    end
end

Boston University
Columbia University
Cornell University
Duke University
Harvard University
London School of Economics and Political Science
Massachusetts Institute of Technology
New York University
Northwestern University
Ohio State University
Pennsylvania State University
Princeton University
Stanford University
University of California Los Angeles (UCLA)
University of California, Berkeley
University of Chicago
University of Maryland
University of Michigan
University of Minnesota, Twin Cities
University of Oxford
University of Pennsylvania
University of Toronto
University of Wisconsin, Madison
Yale University


The estimated allocation is given by the matrix `p2`.  The next section creates a latex table which gives the current adjacency matrix fully labelled.

Labels are created using `ordered_count` which lists the number of institutions that are included in each tier.

In [65]:
# define row titles for the adjacency matrix, include institution counts
names = []

for i in 1:cp["number_of_types"] + cp["number_of_sinks"]
    if i <= cp["number_of_types"]
        push!(names,string("TIER", i, " (", ordered_counts[i]," insts)"))
    else
        push!(names, string(cp["labels"][i-cp["number_of_types"]], " (", ordered_counts[i]," insts)"))
    end
#counter\
end
names

11-element Vector{Any}:
 "TIER1 (24 insts)"
 "TIER2 (77 insts)"
 "TIER3 (198 insts)"
 "TIER4 (345 insts)"
 "TIER5 (621 insts)"
 "Public Sector (226 insts)"
 "Private Sector (335 insts)"
 "Postdocs (801 insts)"
 "Lecturers (614 insts)"
 "Other Groups (674 insts)"
 "Teaching Universities (814 insts)"

In [76]:
save(files_path*"/row_names.jld", "names", names)

In [68]:
SBM_flexible.nice_adjacency_table(p2, names)

┌───────────────────────────────────┬────────┬────────┬────────┬────────┬────────┬────────────┐
│                                   │ Tier 1 │ Tier 2 │ Tier 3 │ Tier 4 │ Tier 5 │ Row Totals │
├───────────────────────────────────┼────────┼────────┼────────┼────────┼────────┼────────────┤
│                  TIER1 (24 insts) │   1638 │    525 │    300 │     62 │     16 │       2541 │
│                  TIER2 (77 insts) │   1444 │   1587 │    566 │    165 │     38 │       3800 │
│                 TIER3 (198 insts) │   1137 │   1486 │   1402 │    221 │     73 │       4319 │
│                 TIER4 (345 insts) │    375 │    692 │    623 │    682 │    104 │       2476 │
│                 TIER5 (621 insts) │     36 │     77 │    107 │     92 │    460 │        772 │
│         Public Sector (226 insts) │    716 │    678 │    396 │    108 │     55 │       1953 │
│        Private Sector (335 insts) │    926 │    549 │    349 │     99 │     54 │       1977 │
│              Postdocs (801 insts) │   

Finally, save the nice table and the placement rates.

In [70]:
SBM_flexible.nice_adjacency_table(p2,names, files_path*"nice_adjacency_table.tex")